# VirtualiZarr → Icechunk → Append Copernicus Data

This notebook demonstrates creating an Icechunk store with virtual references to multiple Copernicus Marine Service files, then appending them along the time dimension.

Based on the NDVI CDR append example from NMFS HackDays 2026.

## Workflow

1. Get URLs for multiple Copernicus NetCDF files from S3
2. Open first file as virtual dataset
3. Write virtual references to Icechunk store
4. Loop: open next file, append to Icechunk along time dimension
5. Commit all changes

**Key Point**: No data is downloaded. We only store virtual references to chunks in the original Copernicus S3 files.

In [ ]:
!pip install -qU icechunk virtualizarr copernicusmarine xarray obstore obspec_utils

In [1]:
import warnings
import shutil
import time
from pathlib import Path

import xarray as xr
import icechunk
from obstore.store import from_url
from virtualizarr import open_virtual_dataset
from virtualizarr.parsers import HDFParser
from obspec_utils.registry import ObjectStoreRegistry

warnings.filterwarnings(
    "ignore",
    message="Numcodecs codecs are not in the Zarr version 3 specification*",
    category=UserWarning,
)

## Step 1: Get S3 URLs for Multiple Files

We'll get a list of consecutive daily files from Copernicus Marine Service.

In [4]:
# Get list of files for several consecutive days
# Using July 2024 as an example
!copernicusmarine get \
  --dataset-id cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D \
  --dataset-version 202603 \
  --filter "*202407*.nc" \
  --create-file-list copernicus_files_multi.txt

INFO - 2026-07-24T15:49:06Z - Selected dataset version: "202603"
INFO - 2026-07-24T15:49:06Z - Selected dataset part: "default"
INFO - 2026-07-24T15:49:07Z - Listing files on remote server...
11it [00:05,  1.98it/s]
{
  "number_of_files_to_download": 0,
  "status": "002",
  "message": "The request created a file list and then stopped."
}


In [6]:
# Read the S3 URLs
with open('copernicus_files_multi.txt', 'r') as f:
    s3_urls = [line.strip() for line in f if line.strip()]

# Sort to ensure chronological order
s3_urls.sort()

print(f"Found {len(s3_urls)} files")
for url in s3_urls:
    print(f"  {Path(url).name}")

Found 31 files
  20240701_cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D.nc
  20240702_cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D.nc
  20240703_cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D.nc
  20240704_cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D.nc
  20240705_cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D.nc
  20240706_cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D.nc
  20240707_cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D.nc
  20240708_cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D.nc
  20240709_cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D.nc
  20240710_cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D.nc
  20240711_cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D.nc
  20240712_cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D.nc
  20240713_cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D.nc
  20240714_cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D.nc
  20240715_cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D.nc
  20240716_cmems_obs-oc_g

## Step 2: Convert to HTTPS URLs

Convert S3 URLs to HTTPS format for cloudferro endpoint.

In [7]:
# Copernicus cloudferro endpoint
COPERNICUS_ENDPOINT = "https://s3.waw3-1.cloudferro.com"

def s3_to_https(s3_url, endpoint=COPERNICUS_ENDPOINT):
    """Convert s3://bucket/path to https://endpoint/bucket/path"""
    if s3_url.startswith('s3://'):
        path = s3_url[5:]  # Remove 's3://'
        return f"{endpoint}/{path}"
    return s3_url

https_urls = [s3_to_https(url) for url in s3_urls]
print(f"Converted {len(https_urls)} URLs to HTTPS format")
print(f"First URL: {https_urls[0]}")

Converted 31 URLs to HTTPS format
First URL: https://s3.waw3-1.cloudferro.com/mdl-native-16/native/OCEANCOLOUR_GLO_BGC_L3_MY_009_103/cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D_202603/2024/07/20240701_cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D.nc


## Step 3: Set up Remote File Access

Configure object store and parser for accessing remote Copernicus files.

In [8]:
# Create object-store handle for remote files
url_prefix = f"{COPERNICUS_ENDPOINT}/"
store = from_url(url_prefix)
registry = ObjectStoreRegistry({url_prefix: store})

# Use HDF parser for NetCDF files
parser = HDFParser()

print(f"✓ Remote storage configured for: {url_prefix}")

✓ Remote storage configured for: https://s3.waw3-1.cloudferro.com/


## Step 4: Create Icechunk Repository

Set up local Icechunk storage with virtual chunk configuration.

In [9]:
# Set up local storage path
repo_path = Path("./copernicus_icechunk_append")
if repo_path.exists():
    shutil.rmtree(repo_path)
    print(f"Cleared existing repo at {repo_path}/")

# Configure virtual chunk container
# This tells Icechunk where the actual data chunks live
config = icechunk.RepositoryConfig.default()
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix=url_prefix,
        store=icechunk.http_store(),
    )
)

# Create local repository
storage = icechunk.local_filesystem_storage(str(repo_path))
repo = icechunk.Repository.create(storage, config)

# Create writable session
session = repo.writable_session("main")

print(f"✓ Created Icechunk repository at {repo_path}")

✓ Created Icechunk repository at copernicus_icechunk_append


  2026-07-24T15:50:31.813245Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:324



## Step 5: Loop Through Files and Append

For each file:
1. Open as virtual dataset (no download)
2. First file: write to Icechunk
3. Subsequent files: append along time dimension

In [11]:
# Process all files
for i, url in enumerate(https_urls[0:2]):
    filename = Path(url).name
    start = time.perf_counter()
    
    print(f"[{i+1}/{len(https_urls)}] Adding {filename}...")
    
    # Open file virtually (no data download)
    vds = open_virtual_dataset(
        url=url,
        parser=parser,
        registry=registry,
        loadable_variables=["time", "latitude", "longitude"],
        decode_times=True,
    )
    
    # First file: create initial dataset
    # Subsequent files: append along time dimension
    if i == 0:
        vds.virtualize.to_icechunk(session.store)
    else:
        vds.virtualize.to_icechunk(session.store, append_dim="time")
    
    elapsed = time.perf_counter() - start
    print(f"  ✓ Finished in {elapsed:.2f} seconds")

print("\n✓ All files processed")

[1/31] Adding 20240701_cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D.nc...
  ✓ Finished in 42.79 seconds
[2/31] Adding 20240702_cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D.nc...
  ✓ Finished in 39.62 seconds

✓ All files processed


## Step 6: Commit Changes

Commit all virtual references to the Icechunk repository.

In [12]:
# Commit all changes
snapshot_id = session.commit(f"Added {len(https_urls)} days of Copernicus chlorophyll data")
print(f"✓ Committed snapshot: {snapshot_id}")

✓ Committed snapshot: H3YB7EZ6EF5E2ETDFV3G


## Step 7: Read and Verify

Open the Icechunk store and verify we have all time steps.
We must provide authorization to access the virtual chunks pointing to Copernicus.

In [14]:
# Tell Icechunk how to authenticate to the underlying data
# For Copernicus cloudferro HTTPS access, we use http_store
credentials = icechunk.containers_credentials({
    url_prefix: icechunk.http_store()
})

# Open the existing Icechunk repository for reading
# Because this dataset contains virtual chunks pointing to Copernicus HTTPS URLs,
# we provide credentials to authorize access to those external files
repo_read = icechunk.Repository.open(
    storage,
    config,
    authorize_virtual_chunk_access=credentials,
)

# Open a read-only session on the main branch
session_read = repo_read.readonly_session("main")

# Open with xarray
ds = xr.open_zarr(session_read.store, consolidated=False)

print("✓ Dataset opened from Icechunk store:")
print(ds)
print(f"\nTime dimension has {len(ds.time)} steps")

IcechunkError:   x a virtual chunk in this repository resolves to the url prefix https://s3.waw3-1.cloudferro.com/, to be able to fetch the chunk you need to authorize the virtual chunk container when you open/
  | create the repository, see https://icechunk.io/en/stable/virtual/
  | 
  | context:
  |    0: icechunk::store::get
  |            with key="lat/c/0" byte_range=From(0)
  |              at icechunk/src/store.rs:183
  | 
  `-> a virtual chunk in this repository resolves to the url prefix https://s3.waw3-1.cloudferro.com/, to be able to fetch the chunk you need to authorize the virtual chunk container when you open/
      create the repository, see https://icechunk.io/en/stable/virtual/
